<a href="https://colab.research.google.com/github/Debdeep17092002/CODSOFT_TASKNO3/blob/main/handwrittentextgeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import argparse
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

In [2]:
FALLBACK_TEXT = (
    "handwriting is the writing done with a pen or pencil. every person has "
    "a style of their own, and the shapes of letters change from one hand to "
    "another. a recurrent neural network can learn these patterns one "
    "character at a time and then produce new text that follows them. "
) * 40


In [3]:
class CharRNN(nn.Module):
    def __init__(self, vocab, emb=64, hidden=256, layers=2, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hidden, layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x, state=None):
        out, state = self.lstm(self.embed(x), state)
        return self.fc(out), state

In [4]:
def load_corpus(path):
    if path and Path(path).exists():
        return Path(path).read_text(encoding="utf-8", errors="ignore").lower()
    print("No corpus given; using the small built-in fallback text.")
    return FALLBACK_TEXT

In [5]:
def make_batches(data, seq_len, batch_size):
    starts = np.random.randint(0, len(data) - seq_len - 1, size=batch_size)
    x = np.stack([data[s:s + seq_len] for s in starts])
    y = np.stack([data[s + 1:s + seq_len + 1] for s in starts])
    return torch.from_numpy(x), torch.from_numpy(y)

In [6]:
def train(text, epochs, seq_len, batch_size, steps_per_epoch, lr, device):
    chars = sorted(set(text))
    stoi = {c: i for i, c in enumerate(chars)}
    data = np.array([stoi[c] for c in text], dtype=np.int64)
    model = CharRNN(len(chars)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(1, epochs + 1):
        total = 0.0
        for _ in range(steps_per_epoch):
            x, y = make_batches(data, seq_len, batch_size)
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = loss_fn(logits.reshape(-1, len(chars)), y.reshape(-1))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item()
        print(f"epoch {ep:3d}/{epochs}  loss {total / steps_per_epoch:.4f}")
    return model, chars

In [7]:
@torch.no_grad()
def generate(model, chars, seed, length, temperature, device):
    model.eval()
    stoi = {c: i for i, c in enumerate(chars)}
    seed = "".join(c for c in seed.lower() if c in stoi) or random.choice(chars)
    x = torch.tensor([[stoi[c] for c in seed]], device=device)
    out, state = list(seed), None
    logits, state = model(x, state)
    for _ in range(length):
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        idx = torch.multinomial(probs, 1)
        out.append(chars[idx.item()])
        logits, state = model(idx, state)
    return "".join(out)

In [8]:
def load_glyphs(csv_path, img_root):
    """Map each character label to a list of image paths."""
    glyphs = {}
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            glyphs.setdefault(row["label"], []).append(Path(img_root) / row["image"])
    return glyphs

In [9]:
def render(text, glyphs, out_path, height=48, line_width=900, space=24):
    """Stitch real handwritten character images into lines of text."""
    from PIL import Image

    def glyph_for(ch):
        for key in (ch, ch.upper()):
            paths = [p for p in glyphs.get(key, []) if p.exists()]
            if paths:
                img = Image.open(random.choice(paths)).convert("L")
                w = max(1, int(img.width * height / img.height))
                return img.resize((w, height))
        return None

    lines, cur, cur_w = [], [], 0
    for ch in text:
        if ch in " \n":
            g, w = None, space
        else:
            g = glyph_for(ch)
            if g is None:
                continue
            w = g.width + 4
        if cur_w + w > line_width:
            lines.append(cur)
            cur, cur_w = [], 0
        cur.append((g, w))
        cur_w += w
    lines.append(cur)

    canvas = Image.new("L", (line_width, (height + 16) * len(lines) + 16), 255)
    for i, line in enumerate(lines):
        x = 8
        for g, w in line:
            if g is not None:
                canvas.paste(g, (x, 8 + i * (height + 16)))
            x += w
    canvas.save(out_path)
    print("saved", out_path)

In [11]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", default=None)
    ap.add_argument("--csv", default="english.csv")
    ap.add_argument("--img_root", default=".")
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--seq_len", type=int, default=100)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--steps", type=int, default=100)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--length", type=int, default=300)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--seed", default="handwriting ")
    ap.add_argument("--out", default="generated.png")
    args, _ = ap.parse_known_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    text = load_corpus(args.corpus)
    model, chars = train(text, args.epochs, args.seq_len, args.batch_size,
                         args.steps, args.lr, device)
    torch.save({"model": model.state_dict(), "chars": chars}, "char_rnn.pt")

    sample = generate(model, chars, args.seed, args.length, args.temperature, device)
    print("\n--- generated text ---\n" + sample)
    Path("generated.txt").write_text(sample)

    glyphs = load_glyphs(args.csv, args.img_root)
    if any(p.exists() for ps in glyphs.values() for p in ps[:1]):
        render(sample, glyphs, args.out)
    else:
        print(f"\nImg/ folder not found under '{args.img_root}', skipping rendering. "
              "Point --img_root at the folder that contains Img/.")


if __name__ == "__main__":
    main()

No corpus given; using the small built-in fallback text.
epoch   1/30  loss 1.1259
epoch   2/30  loss 0.0556
epoch   3/30  loss 0.0402
epoch   4/30  loss 0.0357
epoch   5/30  loss 0.0335
epoch   6/30  loss 0.0325
epoch   7/30  loss 0.0318
epoch   8/30  loss 0.0315
epoch   9/30  loss 0.0310
epoch  10/30  loss 0.0310
epoch  11/30  loss 0.0307
epoch  12/30  loss 0.0305
epoch  13/30  loss 0.0304
epoch  14/30  loss 0.0300
epoch  15/30  loss 0.0301
epoch  16/30  loss 0.0299
epoch  17/30  loss 0.0300
epoch  18/30  loss 0.0299
epoch  19/30  loss 0.0298
epoch  20/30  loss 0.0297
epoch  21/30  loss 0.0297
epoch  22/30  loss 0.0296
epoch  23/30  loss 0.0297
epoch  24/30  loss 0.0298
epoch  25/30  loss 0.0297
epoch  26/30  loss 0.0296
epoch  27/30  loss 0.0297
epoch  28/30  loss 0.0295
epoch  29/30  loss 0.0294
epoch  30/30  loss 0.0295

--- generated text ---
handwriting is the writing done with a pen or pencil. every person has a style of their own, and the shapes of letters change from one hand